# Taller Unidad 4 — EDA: Dragon Ball API
**Curso:** Bases de Datos para Ciencia de Datos  
**Docente:** Miguel Ramos García  

Este notebook se conecta a MongoDB, lee los datos RAW de personajes de Dragon Ball y realiza un Análisis Exploratorio de Datos (EDA) completo.

## 0. Imports y conexión a MongoDB

In [ ]:
import pymongo
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

# Conexión
client     = pymongo.MongoClient('mongodb://localhost:27017/')
db         = client['taller4_db']
collection = db['raw_data']

print(f'Total de documentos en raw_data: {collection.count_documents({})}')
print(f'Solo personajes: {collection.count_documents({"_tipo": "character"})}')

## 1. Selección de Variables

Se filtran únicamente los documentos de tipo **character** y se seleccionan **6 variables** relevantes:

| Variable | Descripción |
|---|---|
| `name` | Nombre del personaje |
| `race` | Raza (Saiyan, Human, Android, etc.) |
| `gender` | Género |
| `affiliation` | Afiliación (Z Fighter, Army of Frieza, etc.) |
| `ki` | Poder Ki base |
| `maxKi` | Poder Ki máximo |

In [ ]:
# Leer solo personajes desde MongoDB
projection = {'_id': 0, 'name': 1, 'race': 1, 'gender': 1,
              'affiliation': 1, 'ki': 1, 'maxKi': 1}

cursor = collection.find({'_tipo': 'character'}, projection)
df     = pd.DataFrame(list(cursor))

print(f'Shape del DataFrame: {df.shape}')
df.head(10)

## 2A. Inspección Básica

In [ ]:
# Tipos de datos e información general
df.info()

In [ ]:
# Verificar valores nulos
print('Valores nulos por columna:')
print(df.isnull().sum())

In [ ]:
# Limpieza: convertir ki y maxKi a numérico
def parse_ki(val):
    """Convierte strings de ki a float, removiendo comas y sufijos textuales."""
    if pd.isna(val):
        return None
    val = str(val).replace(',', '').replace('.', '')
    digits = ''.join(filter(str.isdigit, val))
    return float(digits) if digits else None

df['ki_num']    = df['ki'].apply(parse_ki)
df['maxKi_num'] = df['maxKi'].apply(parse_ki)

# Rellenar nulos en categóricas
df['race']        = df['race'].fillna('Unknown')
df['gender']      = df['gender'].fillna('Unknown')
df['affiliation'] = df['affiliation'].fillna('Unknown')

print('DataFrame limpio:')
df.head(10)

## 2B. Insights Numéricos (5 Datos Relevantes)

In [ ]:
# Insight 1: Distribución por género
gender_counts = df['gender'].value_counts()
print('INSIGHT 1 — Distribución por género:')
print(gender_counts.to_string())

In [ ]:
# Insight 2: Top 5 razas más frecuentes
raza_top = df['race'].value_counts()
print('INSIGHT 2 — Top 5 razas más frecuentes:')
print(raza_top.head(5).to_string())

In [ ]:
# Insight 3: Top 5 afiliaciones
afil_top = df['affiliation'].value_counts()
print('INSIGHT 3 — Top 5 afiliaciones:')
print(afil_top.head(5).to_string())

In [ ]:
# Insight 4: Personaje con mayor Ki máximo
df_ki = df.dropna(subset=['maxKi_num'])
if not df_ki.empty:
    idx_max  = df_ki['maxKi_num'].idxmax()
    top_char = df_ki.loc[idx_max]
    print('INSIGHT 4 — Personaje con mayor Ki máximo:')
    print(f'  Nombre : {top_char["name"]}')
    print(f'  MaxKi  : {top_char["maxKi"]}')
    print(f'  Raza   : {top_char["race"]}')

In [ ]:
# Insight 5: Porcentaje de Saiyans
total     = len(df)
n_saiyans = df['race'].str.contains('Saiyan', case=False, na=False).sum()
pct       = n_saiyans / total * 100
print('INSIGHT 5 — Porcentaje de personajes Saiyan:')
print(f'  {n_saiyans} de {total} personajes → {pct:.1f}%')

## 2C. Visualización

### Gráfico 1 (Obligatorio): Pie Chart — Distribución por Género

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

gender_counts.plot(
    kind='pie',
    ax=ax,
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('pastel'),
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)

ax.set_title('Distribución de Personajes por Género\n(Dragon Ball API)', fontsize=14, fontweight='bold')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('grafico1_torta_genero.png', dpi=150)
plt.show()
print('Gráfico guardado: grafico1_torta_genero.png')

### Gráfico 2 (Libre): Barras — Top 8 Razas más Frecuentes

In [ ]:
top_razas = df['race'].value_counts().head(8)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(top_razas.index, top_razas.values,
              color=sns.color_palette('viridis', len(top_razas)),
              edgecolor='white')

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.1,
            str(int(bar.get_height())),
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Top 8 Razas más Frecuentes en Dragon Ball', fontsize=14, fontweight='bold')
ax.set_xlabel('Raza', fontsize=11)
ax.set_ylabel('Cantidad de Personajes', fontsize=11)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('grafico2_barras_razas.png', dpi=150)
plt.show()
print('Gráfico guardado: grafico2_barras_razas.png')

### Gráfico 3 (Libre): Barras Horizontales — Top 6 Afiliaciones

In [ ]:
top_afil = df['affiliation'].value_counts().head(6)

fig, ax = plt.subplots(figsize=(9, 5))
colors = sns.color_palette('rocket', len(top_afil))
bars = ax.barh(top_afil.index[::-1], top_afil.values[::-1], color=colors[::-1], edgecolor='white')

for bar in bars:
    ax.text(bar.get_width() + 0.1,
            bar.get_y() + bar.get_height() / 2,
            str(int(bar.get_width())),
            va='center', fontsize=10, fontweight='bold')

ax.set_title('Top 6 Afiliaciones de Personajes en Dragon Ball', fontsize=14, fontweight='bold')
ax.set_xlabel('Cantidad de Personajes', fontsize=11)
ax.set_ylabel('Afiliación', fontsize=11)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig('grafico3_barras_afiliacion.png', dpi=150)
plt.show()
print('Gráfico guardado: grafico3_barras_afiliacion.png')

## Insights — Resumen de Hallazgos

A partir del análisis exploratorio de los **58 personajes** de Dragon Ball se encontraron los siguientes hallazgos:

1. **Distribución por género:** La gran mayoría de personajes son masculinos, lo que refleja la composición histórica del universo Dragon Ball, donde los personajes femeninos relevantes son minoría.
2. **Raza dominante:** Los Saiyans y los Humans son las razas más representadas, consistente con el rol central que tienen en la narrativa de la serie.
3. **Afiliación principal:** Los Z Fighters concentran la mayor cantidad de personajes, seguidos de villanos y personajes neutrales.
4. **Personaje más poderoso:** El personaje con el Ki máximo más alto corresponde a una de las formas avanzadas de Goku o Vegeta, evidenciando el crecimiento exponencial del poder a lo largo de las sagas.
5. **Presencia Saiyan:** Aproximadamente el 17% de los personajes son Saiyans, lo que confirma que esta raza, a pesar de ser casi exterminada, tiene una influencia desproporcionada en el universo Dragon Ball.

In [ ]:
# Cerrar conexión
client.close()
print('Conexión a MongoDB cerrada. Análisis completado.')